<a href="https://colab.research.google.com/github/ngtan369/Randomized-SVD-in-compress-Data/blob/main/svd_compressData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">
  <img src="https://oisp.hcmut.edu.vn/en/wp-content/uploads/2017/10/HCMUT-BachKhoa-Logo-480x487.png" alt="HCMUT Logo" width="150">
  
***Phân tích Randomized SVD (Truncated SVD) và Ứng dụng trong nén dữ liệu****

</div>

In [1]:
print("Hello Linear Algebra Course !!")

Hello Linear Algebra Course !!


## 1. Cơ sở lý thuyết của SVD và Compact SVD

### Singular Value Decomposition (SVD)

Phân tích giá trị kỳ dị (SVD) là một kỹ thuật phân tách ma trận mạnh mẽ, tổng quát hóa khái niệm phân tích riêng (eigen-decomposition) cho các ma trận không vuông. Đối với một ma trận thực $A$ kích thước $m \times n$, SVD phân tách $A$ thành ba ma trận:

$$A = U \Sigma V^T$$

Trong đó:
- $U$ là ma trận trực giao $m \times m$ (`U U^T = I`), các cột của $U$ được gọi là **vector kỳ dị trái**.
- $\Sigma$ (Sigma) là ma trận đường chéo $m \times n$ chứa các **giá trị kỳ dị** không âm, theo thứ tự giảm dần trên đường chéo chính: $\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_{\min(m,n)} \ge 0$. Các phần tử ngoài đường chéo bằng 0.
- $V$ là ma trận trực giao $n \times n$ (`V V^T = I`), các cột của $V$ được gọi là **vector kỳ dị phải**.

SVD luôn tồn tại cho mọi ma trận, bất kể kích thước hoặc hạng của nó. Nó cung cấp một cái nhìn sâu sắc về cấu trúc nền tảng của ma trận, chẳng hạn như hạng, không gian con hàng và không gian con cột.

### Compact SVD (Truncated SVD hoặc Reduced SVD)

Khi hạng $r$ của ma trận $A$ nhỏ hơn $\min(m,n)$, hoặc khi chúng ta chỉ quan tâm đến các thành phần đóng góp chính, chúng ta có thể sử dụng Compact SVD. Thay vì lưu trữ toàn bộ các ma trận $U$, $\Sigma$, và $V$, chúng ta chỉ giữ lại $r$ cột đầu tiên của $U$, $r$ giá trị kỳ dị đầu tiên của $\Sigma$, và $r$ cột đầu tiên của $V$.

Trong trường hợp này, $\Sigma$ trở thành một ma trận đường chéo $r \times r$, và các ma trận $U$ và $V$ có kích thước tương ứng $m \times r$ và $n \times r$. Công thức Compact SVD là:

$$A = U_r \Sigma_r V_r^T$$

Với $U_r$ là ma trận $m \times r$, $\Sigma_r$ là ma trận đường chéo $r \times r$, và $V_r$ là ma trận $n \times r$. Compact SVD rất hữu ích cho việc giảm kích thước và loại bỏ nhiễu, vì các giá trị kỳ dị nhỏ thường tương ứng với nhiễu hoặc các thành phần ít quan trọng của dữ liệu.

## 2. Cơ sở lý thuyết của Randomized SVD (chỉ tính k giá trị kỳ dị lớn nhất)

Đối với các ma trận rất lớn, việc tính toán SVD đầy đủ hoặc thậm chí Compact SVD có thể rất tốn kém về mặt tính toán và bộ nhớ. Randomized SVD là một phương pháp hiệu quả để xấp xỉ các giá trị kỳ dị và vector kỳ dị lớn nhất của một ma trận lớn. Ý tưởng chính là sử dụng các phép chiếu ngẫu nhiên để giảm kích thước của ma trận gốc, sau đó thực hiện SVD trên ma trận nhỏ hơn này.

**Các bước cơ bản của Randomized SVD để tìm $k$ giá trị kỳ dị lớn nhất:**

1.  **Bước 1: Chiếu ngẫu nhiên (Randomized Projection)**
    *   Tạo một ma trận ngẫu nhiên $G$ có kích thước $n \times (k+p)$, trong đó $k$ là số giá trị kỳ dị chúng ta muốn giữ lại và $p$ là một số nguyên nhỏ (thường từ 5 đến 10) để "over-sampling" nhằm tăng độ chính xác của xấp xỉ.
    *   Tính toán ma trận $Y = A G$. Ma trận $Y$ có kích thước $m \times (k+p)$ và chứa thông tin quan trọng về không gian con của $A$ với $k+p$ chiều.

2.  **Bước 2: Chuẩn hóa trực giao (Orthogonalization)**
    *   Thực hiện phân tích QR trên ma trận $Y$ để tìm một cơ sở trực giao cho không gian cột của $Y$. Điều này cho ra ma trận $Q$ có kích thước $m \times (k+p)$, với các cột trực giao.

3.  **Bước 3: Chiếu trở lại không gian nhỏ (Reduced SVD)**
    *   Tính ma trận $B = Q^T A$. Ma trận $B$ có kích thước $(k+p) \times n$, nhỏ hơn nhiều so với $A$.
    *   Thực hiện SVD trên ma trận nhỏ $B$: $B = \hat{U} \Sigma V^T$.

4.  **Bước 4: Xấp xỉ SVD của A (Approximate SVD of A)**
    *   Các vector kỳ dị trái của $A$ được xấp xỉ bởi $U = Q \hat{U}$.
    *   Các giá trị kỳ dị $\Sigma$ và vector kỳ dị phải $V$ được lấy trực tiếp từ SVD của $B$.

Kết quả là $A \approx U \Sigma V^T$, trong đó $U$ có kích thước $m \times (k+p)$, $\Sigma$ là ma trận đường chéo $(k+p) \times (k+p)$, và $V$ có kích thước $n \times (k+p)$. Các giá trị và vector kỳ dị được sắp xếp theo thứ tự giảm dần, cho phép chúng ta chọn $k$ thành phần hàng đầu.

## 3. Ý nghĩa của năng lượng dữ liệu và các giá trị kỳ dị

Trong bối cảnh SVD, **"năng lượng dữ liệu"** thường được hiểu là tổng bình phương của tất cả các giá trị trong ma trận, tương đương với bình phương của chuẩn Frobenius của ma trận:

$$ \|A\|_F^2 = \sum_{i=1}^m \sum_{j=1}^n A_{ij}^2 $$

Một tính chất quan trọng của SVD là tổng bình phương của các giá trị kỳ dị cũng bằng bình phương của chuẩn Frobenius:

$$ \|A\|_F^2 = \sum_{i=1}^{\min(m,n)} \sigma_i^2 $$

Các **giá trị kỳ dị** $\sigma_i$ đo lường mức độ "quan trọng" của từng cặp vector kỳ dị (trái và phải) trong việc tái tạo lại ma trận gốc. Các giá trị kỳ dị lớn hơn tương ứng với các thành phần dữ liệu mang nhiều thông tin hơn hoặc có phương sai lớn hơn. Các giá trị kỳ dị nhỏ hơn thường tương ứng với nhiễu hoặc các chi tiết ít quan trọng.

### Tiêu chuẩn chọn $k$ theo tỉ lệ năng lượng

Để nén dữ liệu mà vẫn giữ lại một phần lớn "năng lượng" (thông tin) của ma trận gốc, chúng ta cần chọn một số hạng $k$ sao cho tổng bình phương của $k$ giá trị kỳ dị lớn nhất đạt một tỉ lệ $\eta$ nhất định so với tổng bình phương của tất cả các giá trị kỳ dị:

$$ \frac{\sum_{i=1}^k \sigma_i^2}{\sum_{i=1}^r \sigma_i^2} \ge \eta $$

Trong đó:
- $k$ là số giá trị kỳ dị (và cặp vector kỳ dị) được giữ lại.
- $\sigma_i$ là giá trị kỳ dị thứ $i$.
- $r$ là hạng của ma trận $A$ (hoặc $\min(m,n)$).
- $\eta$ là tỉ lệ năng lượng mong muốn (ví dụ: 0.95 cho 95% năng lượng).

Tiêu chuẩn này giúp chúng ta xác định số lượng thành phần tối thiểu cần thiết để tái tạo ma trận gốc với một mức độ chính xác mong muốn, từ đó đạt được hiệu quả nén.

# II. Hiện thực

## 4. Thu thập và Tiền xử lý Dữ liệu

Để minh họa Randomized SVD trên một ma trận kích thước lớn, chúng ta sẽ sử dụng bộ dữ liệu **MovieLens 25M Dataset** từ GroupLens Research. Đây là một bộ dữ liệu phổ biến cho các hệ thống khuyến nghị, chứa 25 triệu xếp hạng và 109.000 tag áp dụng cho 62.000 bộ phim bởi 162.000 người dùng.

*   **Nguồn dữ liệu:** [Kaggle - MovieLens 25M Dataset](https://www.kaggle.com/datasets/grouplens/movielens-25m-dataset)


In [3]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile

import warnings
warnings.filterwarnings('ignore')

In [4]:
#importing the movie lens dataset directly to colab
!wget --no-check-certificate https://files.grouplens.org/datasets/movielens/ml-25m.zip

--2026-04-29 04:55:24--  https://files.grouplens.org/datasets/movielens/ml-25m.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 261978986 (250M) [application/zip]
Saving to: ‘ml-25m.zip’

ml-25m.zip          100%[===================>] 249.84M  45.0MB/s    in 5.9s    

2026-04-29 04:55:30 (42.3 MB/s) - ‘ml-25m.zip’ saved [261978986/261978986]



In [6]:
local_zip = './ml-25m.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('/')
zip_ref.close()

Đọc dữ liệu

In [7]:
%%time

#consisit of 25M ratings
rating_df = pd.read_csv('/ml-25m/ratings.csv')

#consist of tags/comments from user
tags_df = pd.read_csv('/ml-25m/tags.csv')

#consist of movie titles
title_df = pd.read_csv('/ml-25m/movies.csv')

CPU times: user 11.9 s, sys: 3.26 s, total: 15.2 s
Wall time: 16 s


*   **Ý nghĩa các biến:** Chúng ta sẽ tập trung vào file `ratings.csv`, bao gồm các cột:
    *   `userId`: ID duy nhất của người dùng.
    *   `movieId`: ID duy nhất của bộ phim.
    *   `rating`: Xếp hạng của người dùng (từ 0.5 đến 5.0).
    *   `timestamp`: Thời gian xếp hạng được tạo (chúng ta sẽ không sử dụng cột này).

**Tiền xử lý để đưa dữ liệu về dạng ma trận số:**

Chúng ta sẽ xây dựng một ma trận xếp hạng người dùng-phim ($A$) nơi các hàng đại diện cho người dùng, các cột đại diện cho phim, và các giá trị trong ma trận là xếp hạng của người dùng cho phim đó. Vì không phải người dùng nào cũng xếp hạng tất cả các phim, ma trận này sẽ rất thưa (sparse). Chúng ta sẽ cần một cách hiệu quả để biểu diễn và xử lý ma trận thưa này.

In [5]:
ratings_df = pd.read_csv('movielens-25m/ratings.csv')

# Hiển thị vài dòng đầu để kiểm tra
print("Dữ liệu ratings.csv ban đầu:")
display(ratings_df.head())

# Tạo ánh xạ ID liên tục cho userId và movieId để xây dựng ma trận
user_ids = ratings_df['userId'].astype('category').cat.codes
movie_ids = ratings_df['movieId'].astype('category').cat.codes

# Lấy số lượng người dùng và phim duy nhất
num_users = len(ratings_df['userId'].unique())
num_movies = len(ratings_df['movieId'].unique())

print(f"Số lượng người dùng duy nhất: {num_users}")
print(f"Số lượng phim duy nhất: {num_movies}")

# Xây dựng ma trận thưa (sparse matrix) người dùng-phim
# Sử dụng Compressed Sparse Row (CSR) matrix để tiết kiệm bộ nhớ
rating_matrix = csr_matrix(
    (ratings_df['rating'], (user_ids, movie_ids)),
    shape=(num_users, num_movies)
)

print(f"Kích thước ma trận xếp hạng (Người dùng x Phim): {rating_matrix.shape}")
print(f"Số lượng phần tử khác 0 trong ma trận: {rating_matrix.nnz}")
print(f"Ma trận có tổng cộng {rating_matrix.shape[0] * rating_matrix.shape[1]} phần tử, \nvới {rating_matrix.nnz} phần tử khác không.")

# Chuyển đổi ma trận thưa thành ma trận dày (dense) nếu kích thước cho phép hoặc cho mục đích thử nghiệm nhỏ
# Cảnh báo: Việc này có thể tốn rất nhiều RAM nếu ma trận quá lớn!
# Ví dụ: matrix_dense = rating_matrix.toarray()

# Đối với các bước sau, chúng ta sẽ làm việc trực tiếp với ma trận thưa hoặc sử dụng các thư viện hỗ trợ ma trận thưa.
# Để đơn giản hóa ví dụ và đảm bảo tính toán, tôi sẽ chuyển đổi một phần nhỏ hoặc xử lý ma trận thưa một cách phù hợp.
# Tuy nhiên, yêu cầu của bài là ma trận lớn, nên chúng ta cần cân nhắc cách làm việc hiệu quả với ma trận thưa.

# Để làm việc với Randomized SVD từ thư viện, thường cần ma trận dense hoặc một số wrapper.
# Để đảm bảo ví dụ chạy được mà không hết RAM với toàn bộ 25M ratings, chúng ta sẽ lấy một phần dữ liệu nhỏ hơn
# hoặc tập trung vào việc chuyển đổi để Randomized SVD có thể xử lý.

# Hiện tại, ma trận `rating_matrix` đã được xây dựng dưới dạng sparse. Chúng ta sẽ sử dụng nó làm ma trận đầu vào `A`.

Đang tải và tiền xử lý dữ liệu MovieLens 25M...


FileNotFoundError: [Errno 2] No such file or directory: 'movielens-25m/ratings.csv'

## 5. Thực hiện Randomized SVD và xác định k theo tỉ lệ năng lượng

Chúng ta sẽ sử dụng thư viện `scikit-learn` để thực hiện Randomized SVD. Hàm `randomized_svd` sẽ trả về $U$, $\Sigma$ (dưới dạng mảng 1D các giá trị kỳ dị), và $V^T$.

Sau đó, chúng ta sẽ viết một hàm để xác định số lượng thành phần $k$ cần thiết để giữ lại một tỉ lệ năng lượng dữ liệu $\eta$ nhất định, dựa trên công thức đã trình bày ở phần lý thuyết:

$$ \frac{\sum_{i=1}^k \sigma_i^2}{\sum_{i=1}^r \sigma_i^2} \ge \eta $$


In [ ]:
from sklearn.utils.extmath import randomized_svd

# Chú ý: rating_matrix là ma trận thưa. randomized_svd có thể làm việc trực tiếp với nó.

# Để tránh lỗi bộ nhớ với ma trận cực lớn khi chuyển sang dense cho các mục đích tính toán
# hoặc khi thực hiện SVD toàn bộ, chúng ta sẽ giới hạn k ban đầu cho Randomized SVD.
# k_components ở đây là số lượng giá trị kỳ dị lớn nhất mà Randomized SVD sẽ tính toán.
# Đây không phải là k cuối cùng được chọn cho nén, mà là một giới hạn trên.
# Chọn k_components = min(num_users, num_movies) // 10 hoặc một giá trị hợp lý khác
# để đảm bảo tính toán khả thi. Ví dụ, lấy 1000 hoặc 2000.

# Lấy min(m,n) để xác định số lượng giá trị kỳ dị tối đa có thể có.
max_singular_values = min(rating_matrix.shape)

# Đặt số lượng thành phần tối đa cho randomized_svd, ví dụ 2000 hoặc nhỏ hơn nếu dataset nhỏ hơn.
# Nếu ma trận quá lớn và 2000 vẫn quá nhiều, có thể giảm xuống.
# Hoặc, chỉ lấy tối đa 500 nếu muốn chạy nhanh.

k_for_randomized_svd = min(2000, max_singular_values - 1) # Đảm bảo k < min(m,n)

print(f"Thực hiện Randomized SVD để tính {k_for_randomized_svd} giá trị kỳ dị lớn nhất...")

# Thực hiện Randomized SVD
# output U, Sigma (1D array of singular values), Vt
U, s, Vt = randomized_svd(rating_matrix,
                          n_components=k_for_randomized_svd,
                          random_state=42)

print("Randomized SVD hoàn tất.")
print(f"Kích thước ma trận U: {U.shape}")
print(f"Kích thước mảng giá trị kỳ dị s: {s.shape}")
print(f"Kích thước ma trận Vt: {Vt.shape}")

# Hàm để xác định k dựa trên tỉ lệ năng lượng
def choose_k_by_energy_retention(singular_values, eta):
    total_energy = np.sum(singular_values**2)
    cumulative_energy = np.cumsum(singular_values**2)

    # Tìm chỉ số k nhỏ nhất sao cho cumulative_energy[k-1] / total_energy >= eta
    k = np.where(cumulative_energy / total_energy >= eta)[0][0] + 1
    return k

# Ví dụ sử dụng: Chọn k để giữ lại 90%, 95% và 99% năng lượng
energy_retention_targets = [0.90, 0.95, 0.99]
selected_ks = {}

print("\nXác định k theo tỉ lệ năng lượng:")
for eta_target in energy_retention_targets:
    k_chosen = choose_k_by_energy_retention(s, eta_target)
    selected_ks[eta_target] = k_chosen
    print(f"Với tỉ lệ năng lượng {eta_target*100:.0f}%, k được chọn là: {k_chosen}")

# Lựa chọn một k để tiếp tục ví dụ (ví dụ: 95% năng lượng)
k_final = selected_ks[0.95]
print(f"\nChọn k_final = {k_final} (tương ứng với 95% năng lượng) cho các bước tiếp theo.")

## 6. Xây dựng ma trận xấp xỉ hạng thấp $A_k$ và đánh giá sai số

Sau khi đã xác định được $k$, chúng ta sẽ xây dựng ma trận xấp xỉ hạng thấp $A_k$ từ các thành phần $U$, $\Sigma$, và $V^T$ đã tính toán.

$$A_k = U_k \Sigma_k V_k^T$$

Trong đó $U_k$, $\Sigma_k$ và $V_k^T$ lần lượt là $k$ cột đầu tiên của $U$, $k$ giá trị kỳ dị lớn nhất (dưới dạng ma trận đường chéo), và $k$ hàng đầu tiên của $V^T$.

Sau đó, chúng ta sẽ đánh giá sai số xấp xỉ bằng chuẩn Frobenius:

$$ \|A - A_k\|_F = \sqrt{\sum_{i=1}^m \sum_{j=1}^n (A_{ij} - (A_k)_{ij})^2 }$$

Một cách hiệu quả để tính toán chuẩn Frobenius của sai số mà không cần xây dựng lại toàn bộ ma trận $A_k$ và trừ $A$ (đặc biệt khi $A$ là ma trận thưa và $A_k$ là dense) là sử dụng tính chất của SVD:

$$ \|A\|_F^2 = \sum_{i=1}^r \sigma_i^2 $$

và

$$ \|A - A_k\|_F^2 = \sum_{i=k+1}^r \sigma_i^2 $$

Do đó,

$$ \|A - A_k\|_F = \sqrt{\sum_{i=k+1}^r \sigma_i^2 }$$

Điều này cho phép chúng ta tính toán sai số chỉ từ các giá trị kỳ dị còn lại, mà không cần phải thực hiện phép nhân ma trận lớn để tái tạo $A_k$ một cách tường minh cho toàn bộ ma trận thưa ban đầu.

In [ ]:
# Xây dựng ma trận xấp xỉ hạng thấp Ak
# Lấy k_final thành phần từ U, s, Vt
U_k = U[:, :k_final]
s_k = np.diag(s[:k_final]) # Chuyển mảng 1D các giá trị kỳ dị thành ma trận đường chéo
Vt_k = Vt[:k_final, :]

# Reconstruct Ak. Lưu ý: kết quả này sẽ là một ma trận dày (dense).
# Do kích thước ma trận ban đầu có thể rất lớn, việc xây dựng Ak đầy đủ
# có thể gây tràn bộ nhớ. Thay vào đó, chúng ta sẽ chủ yếu làm việc với U_k, s_k, Vt_k
# như là dữ liệu nén.

# Nếu muốn tái tạo A_k để kiểm tra (chỉ cho ma trận nhỏ hơn hoặc để hiểu rõ):
# A_k_dense = U_k @ s_k @ Vt_k
# print(f"Kích thước ma trận xấp xỉ A_k_dense (nếu được tái tạo): {A_k_dense.shape}")

print(f"\nĐã trích xuất các thành phần U_k, Sigma_k, V_k^T với k = {k_final}")
print(f"Kích thước U_k: {U_k.shape}")
print(f"Kích thước Sigma_k (ma trận đường chéo): {s_k.shape}")
print(f"Kích thước V_k^T: {Vt_k.shape}")

# Đánh giá sai số xấp xỉ theo chuẩn Frobenius
# Sử dụng tính chất ||A - Ak||_F^2 = sum(sigma_i^2 for i > k)

# Tổng bình phương của tất cả các giá trị kỳ dị (tổng năng lượng)
total_energy_sq = np.sum(s**2)

# Tổng bình phương của các giá trị kỳ dị được giữ lại (năng lượng giữ lại)
retained_energy_sq = np.sum(s[:k_final]**2)

# Tổng bình phương của các giá trị kỳ dị bị loại bỏ (năng lượng mất đi)
lost_energy_sq = total_energy_sq - retained_energy_sq

# Sai số chuẩn Frobenius
frobenius_error = np.sqrt(lost_energy_sq)

print(f"\nTổng năng lượng ban đầu (chuẩn Frobenius squared của A): {total_energy_sq:.2f}")
print(f"Năng lượng giữ lại với k={k_final}: {retained_energy_sq:.2f} ({retained_energy_sq/total_energy_sq*100:.2f}%)")
print(f"Sai số xấp xỉ (chuẩn Frobenius ||A - Ak||_F): {frobenius_error:.2f}")

## 7. So sánh dung lượng lưu trữ

Trong biểu diễn gốc, ma trận $A$ là một ma trận thưa. Nó chỉ lưu trữ các phần tử khác 0 cùng với chỉ số của chúng.

Sau khi nén bằng Randomized SVD, chúng ta lưu trữ ba ma trận nhỏ hơn: $U_k$, $\Sigma_k$ (dưới dạng mảng 1D các giá trị kỳ dị), và $V_k^T$. Mặc dù $U_k$ và $V_k^T$ là các ma trận dày, tổng số phần tử của chúng có thể nhỏ hơn đáng kể so với ma trận gốc, đặc biệt khi $k$ nhỏ.

### Tính toán dung lượng lưu trữ (số lượng phần tử)

*   **Ma trận gốc $A$ (thưa):** Số lượng phần tử khác không (`rating_matrix.nnz`). Mỗi phần tử này cần lưu trữ giá trị và hai chỉ số (hàng, cột). Trong thực tế, các thư viện ma trận thưa tối ưu hóa việc này, nhưng số lượng phần tử khác không là một thước đo tốt về kích thước dữ liệu thực tế.
*   **Ma trận nén ($U_k, \Sigma_k, V_k^T$):**
    *   $U_k$: $m \times k$ phần tử.
    *   $\Sigma_k$: $k$ phần tử (vì nó là đường chéo, chỉ cần lưu các giá trị đường chéo).
    *   $V_k^T$: $k \times n$ phần tử.

Tổng số phần tử cho biểu diễn nén: $m \times k + k + k \times n = k(m + n + 1)$.

Chúng ta sẽ so sánh tổng số phần tử này.

In [ ]:
# Số lượng phần tử khác 0 trong ma trận gốc A (thưa)
original_nnz = rating_matrix.nnz
print(f"Số lượng phần tử khác 0 trong ma trận gốc A: {original_nnz} phần tử")

# Kích thước ma trận gốc
num_users, num_movies = rating_matrix.shape

# Số lượng phần tử trong các ma trận nén (U_k, s_k, Vt_k)
compressed_elements = (num_users * k_final) + k_final + (k_final * num_movies)

print(f"Số lượng phần tử của biểu diễn nén (U_k, Sigma_k, V_k^T): {compressed_elements} phần tử")

# So sánh tỉ lệ nén
compression_ratio = original_nnz / compressed_elements
print(f"Tỉ lệ nén (original_nnz / compressed_elements): {compression_ratio:.2f} lần")

# Giả định lưu trữ mỗi số float là 8 byte (double precision)
# Ước tính dung lượng lưu trữ
original_storage_estimate_bytes = original_nnz * (8 + 4 + 4) # Giá trị + chỉ số hàng + chỉ số cột (ước tính)
compressed_storage_estimate_bytes = compressed_elements * 8 # Mỗi phần tử là một float64

print(f"\nƯớc tính dung lượng lưu trữ ban đầu (bytes, chỉ các phần tử khác 0): {original_nnz * 8} bytes (nếu chỉ lưu giá trị)")
print(f"Ước tính dung lượng lưu trữ nén (bytes): {compressed_storage_estimate_bytes} bytes")
print(f"Giảm dung lượng: {((original_nnz * 8 - compressed_storage_estimate_bytes) / (original_nnz * 8)) * 100:.2f}%")
